# Financial Services Executive Risk Dashboard — AIDP

Reads `finance.gold.transaction_analytics_gold` and `finance.gold.fraud_detection_model_gold`
created by `financial_services_medallion_architecture_demo.ipynb`. Run that notebook first.
This notebook reads Gold only, creates no source data, and does not change medallion tables.

**Decision:** Where should fraud-operations leaders focus analyst investigation this month?
**Scope:** synthetic account activity, transaction-risk flags, gross exposure, concentration and evidence.
Amounts are **assumed USD**, as requested; the source has no currency column. No FX conversion is applied.
These are risk-screening signals, not confirmed fraud, realized loss, account balances, revenue or EBITDA.
Source locations remain the original US/Online/ATM labels; currency labeling does not localize the dataset.

The source generator uses random data over 2024 and no fixed seed. Your totals vary by run.
The default is the latest **observed** month, with no claim of calendar completeness or real-time freshness.

## 1. Setup and reporting scope
Uses the existing AIDP Spark session, Matplotlib and IPython. No Pandas/Arrow conversion or LLM is required.
Account filters apply to every view, chart, denominator and brief. Monthly active accounts are not additive over months.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.dates import DateFormatter
from IPython.display import HTML, Markdown, display

CATALOG = "finance"
REPORT_MONTH = None                 # None = latest observed; or '2024-06'.
ACCOUNT_IDS = []                     # [] = all; e.g. ['ACC00000001'].
TREND_MONTHS = 12
TOP_N = 10
ENABLE_ENTERPRISE_LLM = False        # Optional enterprise_llm(prompt) callable; never required.
plt.rcParams.update({"figure.figsize": (11, 4.5), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.2})
COLORS = {"activity": "#245b73", "flag": "#ad6424", "neutral": "#606c72"}
display(HTML("<style>th,td{padding:9px 12px;text-align:left;border-bottom:1px solid #ddd}th{background:#eef3f4}table{font-size:13px}</style>"))

In [ ]:
"""Shared source embedded in both self-contained AIDP notebooks by build_package.py."""
import hashlib
import json
import re
from datetime import datetime
from html import escape

from pyspark.sql import functions as F
from pyspark.sql import types as T

RULE_VERSION = "FS-TRIAGE-1.1"
RISK_THRESHOLD = 60                 # Matches the supplied Gold construction: strictly greater.
HIGH_VALUE_USD = 5000.0             # Fictional demo policy; source has no currency column.
P1_ACCOUNT_EXPOSURE_USD = 10000.0
VERY_HIGH_SCORE = 85
CURRENCY_ASSUMPTION = "USD"

SOURCE_CONTRACTS = {
    "transaction_analytics_gold": {
        "account_id", "month_year", "total_transactions", "total_amount", "avg_amount",
        "avg_risk_score", "high_risk_transactions", "transaction_types", "merchant_categories",
    },
    "fraud_detection_model_gold": {
        "account_id", "transaction_date", "transaction_type", "amount", "merchant_category",
        "location", "risk_score", "month", "day_of_week", "hour", "is_fraud",
    },
}


def read_gold(spark, catalog):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog):
        raise ValueError("CATALOG must be a simple catalog identifier")
    frames = {}
    for short, required in SOURCE_CONTRACTS.items():
        name = f"{catalog}.gold.{short}"
        try:
            df = spark.table(name)
        except Exception as exc:
            raise RuntimeError(f"Cannot read {name}; run the financial-services medallion notebook and check grants") from exc
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"{name}: missing columns {sorted(missing)}; inspected actual write schema, not just DDL")
        for field in df.schema.fields:
            if field.name in {"amount", "risk_score", "total_transactions", "total_amount", "avg_amount", "avg_risk_score", "high_risk_transactions", "is_fraud"}:
                if not isinstance(field.dataType, T.NumericType):
                    raise ValueError(f"{name}.{field.name} must be numeric, found {field.dataType}")
        if not df.take(1):
            raise ValueError(f"{name} is empty")
        frames[short] = df
    tx, monthly = frames["fraud_detection_model_gold"], frames["transaction_analytics_gold"]
    if not isinstance(tx.schema["transaction_date"].dataType, T.TimestampType):
        raise ValueError("transaction_date must be a Spark timestamp")
    bad_tx = tx.filter(
        F.col("account_id").isNull() | ~F.col("account_id").rlike(r"^ACC[0-9]{8}$") |
        F.col("transaction_date").isNull() | F.col("amount").isNull() |
        F.isnan(F.col("amount").cast("double")) | (F.abs(F.col("amount")) == float("inf")) |
        F.col("risk_score").isNull() | ~F.col("risk_score").between(0, 100) |
        F.isnan(F.col("risk_score").cast("double")) | F.col("is_fraud").isNull() |
        (F.col("is_fraud") != F.when(F.col("risk_score") > RISK_THRESHOLD, 1).otherwise(0))
    )
    if bad_tx.take(1):
        raise ValueError("Gold transaction contract failed: keys, amounts, risk range or synthetic is_fraud label")
    if monthly.filter(
        F.col("account_id").isNull() | F.col("month_year").isNull() |
        ~F.col("month_year").rlike(r"^[0-9]{4}-(0[1-9]|1[0-2])$") |
        F.col("total_transactions").isNull() | (F.col("total_transactions") <= 0) |
        F.col("high_risk_transactions").isNull() |
        ~F.col("high_risk_transactions").between(0, F.col("total_transactions")) |
        F.col("total_amount").isNull() | F.col("avg_amount").isNull() |
        F.col("avg_risk_score").isNull() | ~F.col("avg_risk_score").between(0, 100) |
        F.isnan(F.col("total_amount").cast("double")) | F.isnan(F.col("avg_amount").cast("double")) |
        F.isnan(F.col("avg_risk_score").cast("double"))
    ).take(1):
        raise ValueError("Gold account-month contract failed: period, keys, counts or measures")
    if monthly.groupBy("account_id", "month_year").count().filter("count > 1").take(1):
        raise ValueError("Duplicate Gold account-month rows; aggregation would double-count")
    tx = (tx.withColumn("month_year", F.date_format("transaction_date", "yyyy-MM"))
          .withColumn("period_key", F.date_format("transaction_date", "yyyyMM").cast("int"))
          .withColumn("gross_amount_usd", F.abs("amount").cast("double"))
          .withColumn("rule_flag", (F.col("risk_score") > RISK_THRESHOLD).cast("int")))
    return tx, monthly


def account_metrics(tx):
    return (tx.groupBy("month_year", "period_key", "account_id").agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").cast("double").alias("net_amount_usd"),
        F.sum("gross_amount_usd").alias("gross_amount_usd"),
        F.sum(F.when(F.col("amount") > 0, F.col("amount")).otherwise(0)).cast("double").alias("positive_amount_usd"),
        F.sum(F.when(F.col("amount") < 0, -F.col("amount")).otherwise(0)).cast("double").alias("negative_magnitude_usd"),
        F.avg("risk_score").alias("avg_risk_score"),
        F.max("risk_score").alias("max_risk_score"),
        F.sum("rule_flag").alias("flagged_count"),
        F.sum(F.when(F.col("rule_flag") == 1, F.col("gross_amount_usd")).otherwise(0)).alias("flagged_gross_usd"),
        F.sum(F.when((F.col("rule_flag") == 1) & (F.col("gross_amount_usd") >= HIGH_VALUE_USD), 1).otherwise(0)).alias("high_value_flagged_count"),
        F.sum(F.when((F.col("risk_score") >= VERY_HIGH_SCORE) & (F.col("gross_amount_usd") >= HIGH_VALUE_USD), 1).otherwise(0)).alias("very_high_value_count"),
    ).withColumn("flagged_rate_pct", 100.0 * F.col("flagged_count") / F.col("transaction_count"))
     .withColumn("priority", F.when((F.col("flagged_gross_usd") >= P1_ACCOUNT_EXPOSURE_USD) |
                                    (F.col("very_high_value_count") > 0), "P1")
                 .when(F.col("flagged_count") > 0, "P2").otherwise("P3")))


def portfolio_metrics(accounts):
    return (accounts.groupBy("month_year", "period_key").agg(
        F.count("*").alias("active_accounts"),
        *[F.sum(c).alias(c) for c in ["transaction_count", "net_amount_usd", "gross_amount_usd",
             "positive_amount_usd", "negative_magnitude_usd", "flagged_count", "flagged_gross_usd", "high_value_flagged_count"]],
        (F.sum(F.col("avg_risk_score") * F.col("transaction_count")) / F.sum("transaction_count")).alias("avg_risk_score"),
        F.sum(F.when(F.col("priority") == "P1", 1).otherwise(0)).alias("p1_accounts"),
    ).withColumn("flagged_rate_pct", 100.0 * F.col("flagged_count") / F.col("transaction_count")))


def reconciliation_checks(tx, monthly):
    expected = account_metrics(tx).select("account_id", "month_year", "transaction_count", "net_amount_usd", "avg_risk_score", "flagged_count")
    joined = expected.alias("e").join(monthly.alias("m"), ["account_id", "month_year"], "full")
    # Currency totals in source may have DECIMAL(15,2) rounding. Risk averages are not rounded.
    bad = joined.filter(
        F.col("e.transaction_count").isNull() | F.col("m.total_transactions").isNull() |
        (F.col("e.transaction_count") != F.col("m.total_transactions")) |
        (F.abs(F.col("e.net_amount_usd") - F.col("m.total_amount")) > 0.02) |
        (F.abs(F.col("e.net_amount_usd") / F.col("e.transaction_count") - F.col("m.avg_amount")) > 0.02) |
        (F.abs(F.col("e.avg_risk_score") - F.col("m.avg_risk_score")) > 1e-8) |
        (F.col("e.flagged_count") != F.col("m.high_risk_transactions"))
    )
    if bad.take(1):
        raise ValueError("Gold tables do not reconcile by account-month; rerun Gold from the same Silver snapshot")
    # Check distributions by category as well as total counts, without exploding both maps together.
    for column, dimension in [("transaction_types", "transaction_type"), ("merchant_categories", "merchant_category")]:
        stored = monthly.select("account_id", "month_year", F.explode_outer(column).alias(dimension, "stored_count"))
        observed = tx.groupBy("account_id", "month_year", dimension).count()
        mismatch = observed.join(stored, ["account_id", "month_year", dimension], "full").filter(
            F.col("count").isNull() | F.col("stored_count").isNull() | (F.col("count") != F.col("stored_count")))
        if mismatch.take(1):
            raise ValueError(f"Gold map {column} does not reconcile to transaction-level Gold")
    return {"account_month_grain": "PASS", "counts_amounts_risk": "PASS", "category_maps": "PASS"}


def content_fingerprint(tx, monthly):
    parts = []
    for df, short in [(tx, "fraud_detection_model_gold"), (monthly, "transaction_analytics_gold")]:
        cols = sorted(SOURCE_CONTRACTS[short])
        # Commutative multiset checksum includes duplicate multiplicity; not a source commit ID.
        row = df.agg(F.count("*").alias("n"), F.sum(
            F.xxhash64(F.to_json(F.struct(*cols))).cast("decimal(38,0)" )).alias("checksum")).first()
        parts.append([short, int(row["n"]), str(row["checksum"])])
    return hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()[:24]


def rows(df, limit=20):
    return [r.asDict(recursive=True) for r in df.limit(limit).collect()]


def usd(value):
    return "—" if value is None else f"USD {float(value):,.2f}"


def table_html(data, columns):
    headers = "".join(f"<th>{escape(label)}</th>" for key, label in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(row.get(key, '—')))}</td>" for key, label in columns) + "</tr>" for row in data)
    return ("<div style='overflow:auto'><table style='border-collapse:collapse;width:100%'><thead><tr>" +
            headers + "</tr></thead><tbody>" + body + "</tbody></table></div>")


## 2. Validate the actual Gold contract
The supplied medallion notebook overwrites the DDL tables with DataFrames that omit several declared timestamp
columns. Validation therefore uses columns actually written. `is_fraud = (risk_score > 60)` is a synthetic
rule label; the saved Gold table contains ML inputs, not predictions or fraud probabilities.

In [ ]:
transactions_all, monthly_all = read_gold(spark, CATALOG)
source_fingerprint_at_start = content_fingerprint(transactions_all, monthly_all)
source_periods = [r["month_year"] for r in transactions_all.select("month_year").distinct().orderBy("month_year").limit(241).collect()]
if len(source_periods) > 240:
    raise ValueError("More than 240 observed months; choose a bounded source window before collecting periods")
if not isinstance(TREND_MONTHS, int) or not 1 <= TREND_MONTHS <= 24:
    raise ValueError("TREND_MONTHS must be between 1 and 24")
if not isinstance(TOP_N, int) or not 1 <= TOP_N <= 50:
    raise ValueError("TOP_N must be between 1 and 50")
if REPORT_MONTH is not None and not re.fullmatch(r"[0-9]{4}-(0[1-9]|1[0-2])", REPORT_MONTH):
    raise ValueError("REPORT_MONTH must be YYYY-MM")
if REPORT_MONTH is not None and REPORT_MONTH not in source_periods:
    raise ValueError(f"Requested month {REPORT_MONTH} unavailable. Observed: {source_periods}")
report_month = REPORT_MONTH or source_periods[-1]
if not isinstance(ACCOUNT_IDS, list) or any(not isinstance(a, str) or not re.fullmatch(r"ACC[0-9]{8}", a) for a in ACCOUNT_IDS):
    raise ValueError("ACCOUNT_IDS must be a list of ACC followed by eight digits")
if ACCOUNT_IDS:
    found = {r["account_id"] for r in transactions_all.filter(F.col("account_id").isin(ACCOUNT_IDS)).select("account_id").distinct().collect()}
    if set(ACCOUNT_IDS) - found:
        raise ValueError(f"Unknown account IDs: {sorted(set(ACCOUNT_IDS) - found)}")
scope_filter = F.col("account_id").isin(ACCOUNT_IDS) if ACCOUNT_IDS else F.lit(True)
window_start = F.add_months(F.to_date(F.lit(report_month + "-01")), -(TREND_MONTHS - 1))
tx = transactions_all.filter(scope_filter & (F.to_date(F.concat(F.col("month_year"), F.lit("-01"))) >= window_start) & (F.col("month_year") <= report_month)).cache()
gold_monthly = monthly_all.filter(scope_filter & (F.to_date(F.concat(F.col("month_year"), F.lit("-01"))) >= window_start) & (F.col("month_year") <= report_month))
snapshot = tx.filter(F.col("month_year") == report_month)
if not snapshot.take(1):
    raise ValueError("Selected accounts have no activity in the requested month; choose another month or account")
quality = reconciliation_checks(tx, gold_monthly)
accounts = account_metrics(tx).cache()
portfolio = portfolio_metrics(accounts).cache()
current = portfolio.filter(F.col("month_year") == report_month).first().asDict()
scope_label = "All accounts" if not ACCOUNT_IDS else ", ".join(ACCOUNT_IDS)
print(f"Period: {report_month} | Scope: {scope_label} | USD assumption, no FX conversion")
print(f"Observed source periods: {source_periods[0]} to {source_periods[-1]} | fingerprint: {source_fingerprint_at_start}")

## 3. Executive snapshot
Gross activity = `SUM(ABS(amount))`; net activity = `SUM(amount)`. Positive and negative amounts follow the
source's sign convention. Transfers have no counterparty or direction field. Flagged gross is the absolute
amount of transactions with score **strictly greater than 60**. It is an investigation exposure proxy.

In [ ]:
cards = [
    ("Transactions", f"{current['transaction_count']:,}", "Selected month and account scope"),
    ("Active accounts", f"{current['active_accounts']:,}", "Distinct accounts in selected month"),
    ("Gross activity · USD", usd(current["gross_amount_usd"]), "Absolute transaction amounts"),
    ("Net activity · USD", usd(current["net_amount_usd"]), "Signed sum; not an account balance"),
    ("Rule-flagged activity", f"{current['flagged_count']:,} · {current['flagged_rate_pct']:.1f}%", "risk_score > 60; not confirmed fraud"),
    ("Flagged gross · USD", usd(current["flagged_gross_usd"]), "Exposure proxy; not loss or savings"),
]
display(HTML("<div style='display:grid;grid-template-columns:repeat(auto-fit,minmax(250px,1fr));gap:12px'>" +
    "".join(f"<div style='padding:16px;border:1px solid #ccd5d8;border-top:4px solid #245b73;border-radius:6px'><div>{escape(label)}</div><strong style='font-size:24px'>{escape(value)}</strong><div style='font-size:12px;margin-top:8px'>{escape(note)}</div></div>" for label, value, note in cards) + "</div>"))
display(Markdown(f"**Priority queue:** {current['p1_accounts']:,} P1 accounts under fictional policy `{RULE_VERSION}`. "
                 f"Average rule score: {current['avg_risk_score']:.2f}/100; this score is not a probability."))

## 4. Activity and risk trends
An absent calendar month is shown as a gap, never silently treated as zero or skipped in a month-over-month comparison.
Prior-month comparison uses the immediately preceding calendar month and the same account scope.

In [ ]:
trend_rows = rows(portfolio.orderBy("month_year"), 24)
calendar = spark.range(TREND_MONTHS).select(F.date_format(F.add_months(F.to_date(F.lit(report_month + "-01")), -F.col("id").cast("int")), "yyyy-MM").alias("month_year"))
trend_rows = rows(calendar.join(portfolio, "month_year", "left").orderBy("month_year"), 24)
dates = [datetime.strptime(r["month_year"], "%Y-%m") for r in trend_rows]
def values(key, divisor=1):
    return [float(r[key]) / divisor if r.get(key) is not None else float("nan") for r in trend_rows]
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(dates, values("transaction_count"), marker="o", color=COLORS["activity"], label="All transactions")
axes[0].plot(dates, values("flagged_count"), marker="s", linestyle="--", color=COLORS["flag"], label="Rule-flagged transactions")
axes[0].set(ylabel="Transactions", title=f"Monthly transaction activity · {scope_label}"); axes[0].legend()
axes[1].plot(dates, values("flagged_rate_pct"), marker="s", color=COLORS["flag"])
axes[1].set(ylabel="Flagged transactions (%)", xlabel="Observed calendar month", title="Rule-flagged rate · same population")
axes[1].set_ylim(0, 100)
tick_dates = [dates[i] for i in sorted(set(range(0, len(dates), max(1, len(dates)//5))) | {len(dates)-1})]
axes[1].set_xticks(tick_dates); axes[1].xaxis.set_major_formatter(DateFormatter("%b %Y"))
fig.autofmt_xdate(rotation=0); fig.tight_layout(); plt.show()
previous_month = spark.range(1).select(F.date_format(F.add_months(F.to_date(F.lit(report_month + "-01")), -1), "yyyy-MM").alias("m")).first()["m"]
previous = portfolio.filter(F.col("month_year") == previous_month).first()
if previous is None:
    display(Markdown(f"Calendar comparison unavailable for {previous_month} in this trend window and scope."))
else:
    delta_pp = current["flagged_rate_pct"] - previous["flagged_rate_pct"]
    display(Markdown(f"{report_month}: flagged rate {current['flagged_rate_pct']:.2f}% versus {previous['flagged_rate_pct']:.2f}% in {previous_month}; change **{delta_pp:+.2f} percentage points**."))

In [ ]:
fig, ax = plt.subplots()
ax.plot(dates, values("gross_amount_usd", 1000000), marker="o", color=COLORS["activity"], label="Gross activity")
ax.plot(dates, values("flagged_gross_usd", 1000000), marker="s", linestyle="--", color=COLORS["flag"], label="Flagged gross")
ax.set(title=f"Monthly gross activity and flagged exposure · {scope_label}", ylabel="USD million", xlabel="Calendar month")
ax.set_ylim(bottom=0); ax.legend()
ax.set_xticks(tick_dates); ax.xaxis.set_major_formatter(DateFormatter("%b %Y"))
fig.autofmt_xdate(rotation=0); fig.tight_layout(); plt.show()
display(Markdown("These measures use transaction magnitudes. They do not measure deposits held, revenue, realized loss or avoided loss."))

## 5. Investigation queue and concentration
P1: flagged gross ≥ USD 10,000 per account-month **or** at least one transaction with score ≥ 85 and
absolute amount ≥ USD 5,000. P2: other accounts with score > 60 activity. P3: no flagged activity.
These fictional demo thresholds prioritize analyst review; they are not regulatory standards or model outputs.

In [ ]:
account_snapshot = accounts.filter(F.col("month_year") == report_month)
priority_rows = rows(account_snapshot.filter(F.col("flagged_count") > 0).orderBy(F.desc("flagged_gross_usd"), F.desc("flagged_count"), "account_id"), TOP_N)
formatted = [{**r, "flagged_gross_display": usd(r["flagged_gross_usd"]), "gross_display": usd(r["gross_amount_usd"]), "rate_display": f"{r['flagged_rate_pct']:.1f}%"} for r in priority_rows]
display(HTML(table_html(formatted, [("account_id","Synthetic account"),("priority","Demo priority"),("transaction_count","Transactions"),("flagged_count","Rule flags"),("flagged_gross_display","Flagged gross USD"),("rate_display","Flag rate"),("high_value_flagged_count","Flags ≥ USD 5,000") ])))
top_exposure = sum(float(r["flagged_gross_usd"]) for r in priority_rows)
concentration_pct = 100 * top_exposure / current["flagged_gross_usd"] if current["flagged_gross_usd"] else None
display(Markdown(f"Showing {len(priority_rows)} accounts ranked by flagged gross. " +
    (f"They represent **{concentration_pct:.1f}%** of this month's flagged gross in the selected scope." if concentration_pct is not None else "No rule-flagged gross exposure in this scope.")))
if priority_rows:
    data = list(reversed(priority_rows))
    fig, ax = plt.subplots(figsize=(11, max(4, len(data)*0.35)))
    ax.barh([r["account_id"] for r in data], [r["flagged_gross_usd"] for r in data], color=COLORS["flag"])
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: f"${value:,.0f}"))
    ax.set(title=f"Top {len(data)} accounts by flagged gross · {report_month}", xlabel="Flagged gross · USD")
    fig.tight_layout(); plt.show()

## 6. Transaction type, location and score distribution
Transaction type is not a channel taxonomy. Locations are source labels, not customer residence or geolocation.
No counterparty, transfer destination, customer KYC, sanctions result or device data exists in these sources.

In [ ]:
mix_tables = {}
for dimension, title in [("transaction_type", "Transaction type"), ("location", "Location label")]:
    mix = snapshot.groupBy(dimension).agg(F.count("*").alias("transactions"), F.sum("rule_flag").alias("flags"),
        F.sum(F.when(F.col("rule_flag") == 1, F.col("gross_amount_usd")).otherwise(0)).alias("flagged_gross_usd"))
    data = rows(mix.orderBy(F.desc("flagged_gross_usd"), dimension), 30)
    mix_tables[dimension] = data
    fig, ax = plt.subplots()
    ax.barh([str(r[dimension]) for r in data][::-1], [r["flagged_gross_usd"] for r in data][::-1], color=COLORS["activity"])
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: f"${value/1000000:,.2f}M"))
    ax.set(title=f"Flagged gross by {title.lower()} · {report_month}", xlabel="Flagged gross · USD million")
    fig.tight_layout(); plt.show()

In [ ]:
band = (F.when(F.col("risk_score") <= 20, "00–20").when(F.col("risk_score") <= 40, "21–40")
        .when(F.col("risk_score") <= 60, "41–60").when(F.col("risk_score") <= 80, "61–80").otherwise("81–100"))
risk_rows = rows(snapshot.withColumn("score_band", band).groupBy("score_band").count().orderBy("score_band"), 5)
fig, ax = plt.subplots()
ax.bar([r["score_band"] for r in risk_rows], [r["count"] for r in risk_rows], color=COLORS["activity"])
ax.set(title=f"Rule-score distribution · {report_month}", xlabel="Source risk score (0–100)", ylabel="Transactions")
fig.tight_layout(); plt.show()
display(Markdown("The generator assigns risk scores randomly. Any apparent pattern is synthetic; this chart does not validate a detection model."))

## 7. Gold-grounded executive brief
Deterministic arithmetic generates the default brief. Optional LLM wording receives only aggregated evidence.
No customer contact, account freeze, case creation, payment intervention or filing is performed.

In [ ]:
leader = priority_rows[0] if priority_rows else None
brief = (f"## Investigation review · {report_month}\n\n"
         f"Scope: {scope_label}. Synthetic amounts assumed USD; no currency conversion.\n\n"
         f"- {current['transaction_count']:,} transactions across {current['active_accounts']:,} active accounts; gross activity {usd(current['gross_amount_usd'])}.\n"
         f"- {current['flagged_count']:,} rule flags ({current['flagged_rate_pct']:.2f}%) represent {usd(current['flagged_gross_usd'])} flagged gross. This is not confirmed fraud or loss.\n"
         f"- {current['p1_accounts']:,} account-months meet fictional P1 review criteria under {RULE_VERSION}.\n")
if leader:
    brief += f"- Review {leader['account_id']} first by exposure: {usd(leader['flagged_gross_usd'])}, {leader['flagged_count']} rule flags; queue priority {leader['priority']}.\n"
brief += ("\n**Next step:** Retrieve the applicable synthetic policy and transaction evidence through the Agent Flow; draft an analyst investigation packet. Human review is required before any action.\n\n"
          "**Evidence:** the two finance Gold tables, scoped as above. Neither model predictions nor confirmed fraud labels are persisted in these tables. AUC from the training notebook is not consumed or certified here.")
display(Markdown(brief))
if ENABLE_ENTERPRISE_LLM:
    callback = globals().get("enterprise_llm")
    if not callable(callback):
        raise RuntimeError("ENABLE_ENTERPRISE_LLM=True requires an authorized enterprise_llm(prompt) callable")
    evidence = json.dumps({"period": report_month, "scope": scope_label, "metrics": current, "source_fingerprint": source_fingerprint_at_start}, default=str)
    response = callback("Rewrite the supplied synthetic aggregate evidence into a USD investigation brief. Preserve numbers, scope, caveats and sources exactly. Do not infer confirmed fraud, loss, causal drivers or probabilities. Evidence: " + evidence)
    display(Markdown("### Optional LLM draft — requires analyst verification\n" + str(response)))

## 8. Reconciliation and handoff
Checks reconcile both Gold tables at account-month grain, validate category maps, verify scope totals and
confirm the source content fingerprint did not change during the run. This fingerprint is a multiset checksum,
not a Delta commit/version or an immutable audit record. Rerun after any source refresh.

In [ ]:
assert abs(current["positive_amount_usd"] - current["negative_magnitude_usd"] - current["net_amount_usd"]) < 0.02
assert 0 <= current["flagged_count"] <= current["transaction_count"]
assert current["flagged_gross_usd"] <= current["gross_amount_usd"] + 0.02
assert abs(sum(float(r["flagged_gross_usd"]) for r in mix_tables["transaction_type"]) - current["flagged_gross_usd"]) < 0.02
source_fingerprint_at_end = content_fingerprint(*read_gold(spark, CATALOG))
assert source_fingerprint_at_start == source_fingerprint_at_end, "Source changed during dashboard run; rerun"
quality.update({"signed_amount_identity": "PASS", "risk_bounds": "PASS", "type_mix_totals": "PASS", "stable_source_content": "PASS"})
display(HTML(table_html([{"check": key, "status": value} for key, value in quality.items()], [("check", "Quality check"), ("status", "Result")])))
print("Dashboard checks passed for this run. Next: financial_services_multi_agent_demo_setup_aidp.ipynb")
tx.unpersist(); accounts.unpersist(); portfolio.unpersist()

# Plain-text proof remains readable if the browser cannot render HTML output frames.
print("USD denomination validation; original numerical amounts preserved, no FX conversion")
print(json.dumps({"period": report_month, "scope": scope_label, "currency_assumption": CURRENCY_ASSUMPTION, "rule_version": RULE_VERSION, "source_fingerprint": source_fingerprint_at_end, "transactions": current["transaction_count"], "active_accounts": current["active_accounts"], "gross_activity_usd": current["gross_amount_usd"], "net_activity_usd": current["net_amount_usd"], "flagged_count": current["flagged_count"], "flagged_gross_usd": current["flagged_gross_usd"], "flagged_rate_pct": current["flagged_rate_pct"], "p1_accounts": current["p1_accounts"], "checks": quality}, indent=2, default=str))

**Source adaptation:** replaces manufacturing P&L with transaction activity; plants/products with account risk;
supplier/inventory panels with type/location exposure; model-probability panels with an honest rule-score view.
It preserves executive cards, trends, ranked actions, deterministic briefing, bounded `.collect()` and reconciliation.
Open `docs/agent_flow_build_guide.html` for the SQL + RAG investigation workflow and manual AIDP acceptance steps.